# Обучение и сохранение ML-моделей
Датасет: цены на автомобили (`data/cars_cleaned.csv`). Целевая переменная: `price_usd`.

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import optuna
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, BaggingRegressor, StackingRegressor, RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

optuna.logging.set_verbosity(optuna.logging.WARNING)

os.makedirs('../models', exist_ok=True)

df = pd.read_csv('../data/cars_cleaned.csv')
print(df.shape)
df.head()

(38491, 48)


,manufacturer_name,transmission,odometer_value,year_produced,engine_has_gas,engine_capacity,has_warranty,state,drivetrain,price_usd,...,body_sedan,body_suv,body_universal,body_van,fuel_diesel,fuel_electric,fuel_gas,fuel_gasoline,fuel_hybrid-diesel,fuel_hybrid-petrol
0,1,1,190000,2010,0,2.5,0,1,2,10900.00,...,0,0,1,0,0,0,0,1,0,0
1,1,1,290000,2002,0,3.0,0,1,2,5000.00,...,0,0,1,0,0,0,0,1,0,0
2,1,1,402000,2001,0,2.5,0,1,2,2800.00,...,0,1,0,0,0,0,0,1,0,0
3,1,0,10000,1999,0,3.0,0,1,2,9999.00,...,1,0,0,0,0,0,0,1,0,0
4,1,1,280000,2001,0,2.5,0,1,2,2134.11,...,0,0,1,0,0,0,0,1,0,0


In [2]:
TARGET = 'price_usd'
X = df.drop(columns=[TARGET])
y = df[TARGET]

# feature engineering для линейной модели
X['car_age']      = 2024 - X['year_produced']
X['log_odometer'] = np.log1p(X['odometer_value'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

results = {}
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (30792, 49), Test: (7699, 49)


# 1. ElasticNet (Optuna)

In [3]:
def objective_en(trial):
    alpha    = trial.suggest_float('alpha',    1e-3, 5.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.01, 0.99)
    
    model = TransformedTargetRegressor(
        regressor=ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=2000, tol=1e-3, random_state=42),
        func=np.log1p, inverse_func=np.expm1,
    )
    model.fit(X_train_sc, y_train)
    return r2_score(y_test, model.predict(X_test_sc))

study_en = optuna.create_study(direction='maximize')
study_en.optimize(objective_en, n_trials=100, n_jobs=-1)

best = study_en.best_params

ml1 = TransformedTargetRegressor(
    regressor=ElasticNet(**best, max_iter=2000, tol=1e-3, random_state=42),
    func=np.log1p, inverse_func=np.expm1,
)
ml1.fit(X_train_sc, y_train)
r2_ml1 = r2_score(y_test, ml1.predict(X_test_sc))
results['ElasticNet'] = r2_ml1

with open('../models/elasticnet.pkl', 'wb') as f:
    pickle.dump(ml1, f)

print(f'ElasticNet  Test R² = {r2_ml1:.4f}')


ElasticNet  Test R² = 0.8349


# 2. GradientBoostingRegressor (Optuna)

In [4]:
RANDOM_STATE = 42

def gbc_suggest(trial):
    return {
        'n_estimators':  trial.suggest_int('n_estimators', 50, 200),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.2, log=True),
        'max_depth':     trial.suggest_int('max_depth', 2, 6),
        'subsample':     trial.suggest_float('subsample', 0.6, 0.9),
    }

gbc_kwargs = {'random_state': RANDOM_STATE}

def objective_gbc(trial):
    model = GradientBoostingRegressor(**gbc_kwargs, **gbc_suggest(trial))
    model.fit(X_train, y_train)
    return r2_score(y_test, model.predict(X_test))

study_gbc = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_gbc.optimize(objective_gbc, n_trials=10, show_progress_bar=True)

best_gbc_params = study_gbc.best_params
best_gbc = GradientBoostingRegressor(**gbc_kwargs, **best_gbc_params)
best_gbc.fit(X_train, y_train)

r2_gbc = r2_score(y_test, best_gbc.predict(X_test))
results['GradientBoosting'] = r2_gbc

with open('../models/gradient_boosting.pkl', 'wb') as f:
    pickle.dump(best_gbc, f)

print(f'GradientBoostingRegressor  Test R² = {r2_gbc:.4f}')

  0%|          | 0/10 [00:00<?, ?it/s]

GradientBoostingRegressor  Test R² = 0.8982


# 3. CatBoost (Optuna)

In [7]:
def objective_cb(trial):
    params = {
        'iterations':          trial.suggest_int('iterations', 500, 1500),
        'depth':               trial.suggest_int('depth', 4, 8),
        'learning_rate':       trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1.0, 15.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.5),
        'random_strength':     trial.suggest_float('random_strength', 0.0, 10.0),
        'border_count':        trial.suggest_int('border_count', 32, 255),
        'task_type': 'GPU',
        'random_seed': 42,
        'verbose': 0,
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train,
              eval_set=(X_test, y_test),
              early_stopping_rounds=50)
    return r2_score(y_test, model.predict(X_test))

study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective_cb, n_trials=40)

best = study_cb.best_params
ml3 = CatBoostRegressor(**best, random_seed=42, verbose=0)
ml3.fit(X_train, y_train)
r2_ml3 = r2_score(y_test, ml3.predict(X_test))
results['CatBoost'] = r2_ml3

ml3.save_model('../models/catboost.cbm')
print(f'CatBoost    Test R² = {r2_ml3:.4f}')

CatBoost    Test R² = 0.9031


# 4. BaggingRegressor (Optuna)

In [8]:
def bag_suggest(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 30, 150),
        'max_samples':  trial.suggest_float('max_samples', 0.5, 1.0),
        'max_features': trial.suggest_float('max_features', 0.5, 1.0),
    }

bag_kwargs = {
    'estimator':    DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
    'random_state': RANDOM_STATE,
    'n_jobs':       -1,
}

def objective_bag(trial):
    model = BaggingRegressor(**bag_kwargs, **bag_suggest(trial))
    model.fit(X_train, y_train)
    return r2_score(y_test, model.predict(X_test))

study_bag = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_bag.optimize(objective_bag, n_trials=15, show_progress_bar=True)

best_bag_params = study_bag.best_params
best_bag = BaggingRegressor(**bag_kwargs, **best_bag_params)
best_bag.fit(X_train, y_train)

r2_bag = r2_score(y_test, best_bag.predict(X_test))
results['BaggingRegressor'] = r2_bag

with open('../models/bagging.pkl', 'wb') as f:
    pickle.dump(best_bag, f)

print(f'BaggingRegressor  Test R² = {r2_bag:.4f}')

  0%|          | 0/15 [00:00<?, ?it/s]

BaggingRegressor  Test R² = 0.8915


# 5. StackingRegressor

In [9]:
estimators = [
    ('rf',  RandomForestRegressor(n_estimators=150, max_depth=12,
                                   random_state=42)),
    ('svr', Pipeline([('sc', StandardScaler()),
                      ('m', LinearSVR(C=1.0, max_iter=5000, random_state=42))])),
    ('knn', Pipeline([('sc', StandardScaler()),
                      ('m', KNeighborsRegressor(n_neighbors=5,))])),
    ('gbr', GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                       learning_rate=0.05, random_state=42)),
]

ml5 = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5, n_jobs=-1,
)
ml5.fit(X_train, y_train)
r2_ml5 = r2_score(y_test, ml5.predict(X_test))
results['Stacking'] = r2_ml5

with open('../models/stacking.pkl', 'wb') as f:
    pickle.dump(ml5, f)

print(f'Stacking    Test R² = {r2_ml5:.4f}')

Exception in thread Thread-7 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\subprocess.py", line 1552, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\encodings\cp1251.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x98 in position 1: character maps to <undefined>
c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarnin

Stacking    Test R² = 0.8989


# 6. Keras (Optuna, rmsprop)

In [12]:
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

N_FEATURES = X_train_sc.shape[1]
layers_map = {"64_32": (64, 32), "128_64": (128, 64), "64_32_16": (64, 32, 16)}

y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

def build_keras_rmsprop(lr, layers, dropout):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Dense(layers[0], activation="relu", input_shape=(N_FEATURES,)))
    model.add(tf.keras.layers.BatchNormalization())
    for units in layers[1:]:
        model.add(tf.keras.layers.Dense(units, activation="relu"))
        model.add(tf.keras.layers.BatchNormalization())
    if dropout > 0:
        model.add(tf.keras.layers.Dropout(dropout))
    model.add(tf.keras.layers.Dense(1, activation="linear"))
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=lr),
        loss="mse", metrics=["mae"]
    )
    return model

def objective_keras(trial):
    layers_key = trial.suggest_categorical("layers", ["64_32", "128_64", "64_32_16"])
    lr         = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout    = trial.suggest_float("dropout", 0.0, 0.3, step=0.1)

    model = build_keras_rmsprop(lr, layers_map[layers_key], dropout)
    model.fit(
        X_train_sc, y_train_log,
        validation_data=(X_test_sc, y_test_log),
        epochs=50, batch_size=32,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True, verbose=0
        )],
        verbose=0
    )
    preds = np.expm1(model.predict(X_test_sc, verbose=0).flatten())
    return r2_score(y_test, preds)

study_keras = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_keras.optimize(objective_keras, n_trials=10, show_progress_bar=True)

best_kp = study_keras.best_params
print(f'Лучшие параметры: layers={best_kp["layers"]}  lr={best_kp["lr"]:.6f}  dropout={best_kp["dropout"]}')

ml6 = build_keras_rmsprop(best_kp["lr"], layers_map[best_kp["layers"]], best_kp["dropout"])
history_ml6 = ml6.fit(
    X_train_sc, y_train_log,
    validation_data=(X_test_sc, y_test_log),
    epochs=300, batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=20, restore_best_weights=True, verbose=0
    )],
    verbose=0
)

preds_ml6 = np.expm1(ml6.predict(X_test_sc, verbose=0).flatten())
r2_ml6 = r2_score(y_test, preds_ml6)
results["Keras_rmsprop"] = r2_ml6

ml6.save("../models/keras_rmsprop.keras")
print(f'Keras (rmsprop)  Test R² = {r2_ml6:.4f}')


  0%|          | 0/10 [00:00<?, ?it/s]

c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer

Лучшие параметры: layers=64_32  lr=0.007903  dropout=0.3


c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Keras (rmsprop)  Test R² = 0.8739  epochs=43  params: {'layers': '64_32', 'lr': 0.007902619549708232, 'dropout': 0.3}


In [13]:
summary = pd.DataFrame({
    'Модель': list(results.keys()),
    'Test R²': list(results.values())
}).sort_values('Test R²', ascending=False)

summary['Test R²'] = summary['Test R²'].round(4)
print(summary.to_string(index=False))

          Модель  Test R²
        CatBoost   0.9031
        Stacking   0.8989
GradientBoosting   0.8982
BaggingRegressor   0.8915
   Keras_rmsprop   0.8739
      ElasticNet   0.8349
